# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:43<00:00, 14.62s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Aorlym P12 12" 48GB RAM 2.4K Tablet for $144 + free shipping\nDetails: Clip the $126.00 off coupon on the page and apply code "TOH7U669" for a savings of $156. Buy Now at Amazon\nFeatures: 12-inch 2.4K 120Hz display 48GB RAM and 128GB storage 10000mAh battery with 18W charging Octa-core CPU with Android 16 Includes keyboard and accessories\nURL: https://www.dealnews.com/Aorlym-P12-12-48-GB-RAM-2-4-K-Tablet-for-144-free-shipping/21815468.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Vonuv 3-in-1 Fast Charger for Apple Devices for $9 + free shipping w/ Prime
Details: Apply coupon code "QQFXQXOT" for a savings of $20. Buy Now at Amazon
Features: 20W USB-C and 18W USB-A ports Magnetic pad for Apple Watch Compatible with iPhone, iPad, AirPods Fast charging for multiple devices Compact and portable for travel
URL: https://www.dealnews.com/Vonuv-3-in-1-Fast-Char

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Anker Solix C1000 is a high-capacity portable power station designed for RV use and outdoor camping. It features a long-life LFP battery rated for 3,000 cycles, provides up to 2,400W output with 11 output ports to power most appliances, and supports up to 600W solar input for fast recharging. The unit charges to 80% in 43 minutes and fully charges in under an hour, includes app monitoring with Bluetooth and Wi‑Fi connectivity, and comes with a 5-year warranty. Model: C1000.', price=429.0, url='https://www.dealnews.com/products/Anker-Solix/Anker-Solix-C1000-1-800-W-Portable-Power-Station/497901.html?iref=rss-c142'), Deal(product_description='Aorlym P12 is a 12-inch Android tablet with a 2.4K, 120Hz display and an octa-core CPU, equipped for heavy multitasking with 48GB of RAM and 128GB of storage. It includes a 10,000mAh battery with 18W charging, runs Android 16, and ships with a keyboard and accessories, making it suitable for productivit

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Anker Solix C1000 is a high-capacity portable power station designed for RV use and outdoor camping. It features a long-life LFP battery rated for 3,000 cycles, provides up to 2,400W output with 11 output ports to power most appliances, and supports up to 600W solar input for fast recharging. The unit charges to 80% in 43 minutes and fully charges in under an hour, includes app monitoring with Bluetooth and Wi‑Fi connectivity, and comes with a 5-year warranty. Model: C1000.
429.0
https://www.dealnews.com/products/Anker-Solix/Anker-Solix-C1000-1-800-W-Portable-Power-Station/497901.html?iref=rss-c142

Aorlym P12 is a 12-inch Android tablet with a 2.4K, 120Hz display and an octa-core CPU, equipped for heavy multitasking with 48GB of RAM and 128GB of storage. It includes a 10,000mAh battery with 18W charging, runs Android 16, and ships with a keyboard and accessories, making it suitable for productivity and media consumption. The large RAM configuration targets power users who run many app

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='288Wh LiFePO4 portable battery designed specifically for use with Starlink Mini; integrates a stand and power delivery in one compact unit. It supports dual charging up to 120W and includes a smart BMS with LED indicators for battery status and health. The package comes with a travel case and necessary accessories for easy transport and setup outdoors or on the go.', price=200.0, url='https://www.dealnews.com/Gendome-288-Wh-Starlink-Mini-Battery-with-Case-for-200-free-shipping/21815471.html?iref=rss-c142'), Deal(product_description='Anker Solix C1000 portable power station with an LFP battery and a high-output inverter designed for RV and outdoor use. It delivers up to 2,400W peak output across 11 ports, accepts up to 600W solar input, and charges to 80% in about 43 minutes (full charge under an hour). The unit includes Bluetooth and Wi‑Fi for app monitoring and is rated for 3,000 battery cycles with a 5-year warranty.', price=429.0, url='

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [20]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [1]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [2]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")